# AR processes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sg
from scipy.signal import hilbert
from scipy.stats import shapiro, probplot
import seaborn as sns

def generate_ar(phi, n, c=0, sigma = 1, seed=None):
    rng = np.random.default_rng(seed)
    p = 1
    x = np.zeros(n)
    eps = rng.normal(0, sigma, n)  # white noise

    x[:p] = rng.normal(0, sigma, p)

    for t in range(2,n):
        x[t] = (phi*x[t-1]) + eps[t]

    return x

# Sampling settings (consistent everywhere)
fs = 270                # Hz
T  = 350             # seconds (longer = more stable estimates)
n  = T * fs          # samples
t  = np.arange(2000) / fs

s = np.random.randint(1, 41)
x1 = generate_ar(0, 2000)
x2 = generate_ar(0.6, 2000)
x3 = generate_ar(0.99, 2000)
x4 = generate_ar(0.999, 2000)

In [ ]:
def fourier_phase_randomized(x:np.ndarray, N:int=1, seed:int=None)->np.ndarray:
    """
    Generate N surrogates of an input signal on the time domain.

    Parameters
    ----------
    x : np.ndarray [n_samples]
        Time series of the original signal.
    N : int
        Number of surrogates to generate.
    seed : int
        Optional seed for the random generator.

    Returns
    -------
    y : np.ndarray[n_samples, N]
        Surrogates of the input signal.
    """

    rng = np.random.default_rng(seed)

    n = len(x)
    half = n // 2
    X_amp = np.abs(np.fft.fft(x))

    Yf = np.zeros((n, N), dtype=complex)
    Yf[0] = X_amp[0] # DC component

    phases = rng.uniform(0, 2*np.pi, size=(half - 1, N))
    Yf[1:half] = X_amp[1:half, None] * np.exp(1j * phases)

    if n % 2 == 0:
        Yf[half] = X_amp[half] # Nyquist frequency
        Yf[half+1:] = np.conj(Yf[1:half][::-1])
    else:
        Yf[half+1:] = np.conj(Yf[1:half+1][::-1])

    y = np.real(np.fft.ifft(Yf, axis=0))

    return y


def IAAFT(x:np.ndarray, N:int=1, n_iter:int=10, seed:int=None)->np.ndarray:
    """
    Generate N IAAFT surrogates for signal x.

    Parameters
    ----------
    x : np.ndarray [n_samples]
        Time series of the original signal.
    N : int
        Number of surrogates.
    n_iter : int
        Number of IAAFT iterations.
    seed : int
        Optional seed for the random generator.

    Returns
    -------
    y : np.ndarray [n_samples, N]
        IAAFT surrogate signals.
    """

    x_sorted = np.sort(x)
    X_amp = np.abs(np.fft.fft(x))

    y = fourier_phase_randomized(x, N, seed)

    for _ in range(n_iter):
        # Impose the exact time-domain amplitude distribution
        idx = np.argsort(y, axis=0)
        for k in range(N):
            y[idx[:, k], k] = x_sorted # sort y and replace with sorted original x

        # Impose Fourier amplitudes but keep phases
        Yf = np.fft.fft(y, axis=0)
        phases = Yf / (np.abs(Yf) + 1e-15)
        Yf = X_amp[:, None] * phases # impose original amplitude spectrum
        y = np.real(np.fft.ifft(Yf, axis=0))

    return y


def randomize_phase_velocity(phase:np.ndarray, N:int=1, seed:int=None, return_idx:bool=False)->np.ndarray:
    """
    Creates surrogates of a phase array by permuting the phase velocities.

    Parameters
    ----------
    phase : np.ndarray [n_samples]
        Hilbert phase.
    N : int
        Number of surrogates to generate.
    seed : int
        Optional seed for the random generator.

    Returns
    -------
    randomized_phase : np.ndarray[n_samples, N]
        Surrogates of the phase.
    """

    rng = np.random.default_rng(seed)
    n = len(phase)

    velocity = np.diff(phase)  # (n-1,)

    idx = np.column_stack([
        rng.permutation(n - 1) for _ in range(N)
    ])

    velocity_perm = velocity[idx]

    randomized_phase = np.zeros((n, N))
    randomized_phase[0, :] = phase[0]
    randomized_phase[1:, :] = phase[0] + np.cumsum(velocity_perm, axis=0)

    if return_idx:
        return randomized_phase, idx
    else:
        return randomized_phase


def hilbert_velocity_randomized(x:np.ndarray, N:int=1, seed:int=None, return_hilbert:bool=False)->np.ndarray:
    """
    Generate N surrogates of an input signal on the Hilbert phase domain.

    Parameters
    ----------
    x : np.ndarray [n_samples]
        Time series of the original signal.
    N : int
        Number of surrogates to generate.
    seed : int
        Optional seed for the random generator.

    Returns
    -------
    y : np.ndarray[n_samples, N]
        Surrogates of the input signal.
    """

    analytical = hilbert(x)
    amplitude = np.abs(analytical)
    phase = np.unwrap(np.angle(analytical))

    randomized_phase = randomize_phase_velocity(phase, N, seed)

    y = amplitude[:, None] * np.cos(randomized_phase)

    if return_hilbert:
        return y, phase, randomized_phase, amplitude
    else:
        return y

def comute_omega(fase_unwrapped, fs):
    dt = 1.0 / fs

    omega = np.diff(fase_unwrapped) / dt
    return omega


def compute_v(fase_unwrapped, fs):
    """
    Calcula el coeficiente de variación de la velocidad de fase (V).

    Args:
        fase_unwrapped (array): La fase desenvuelta de la señal.
        fs (int): Frecuencia de muestreo en Hz.

    Returns:
        dict: Contiene los valores de V, M y S.
    """
    dt = 1.0 / fs

    omega = np.diff(fase_unwrapped) / dt

    M = np.mean(omega)

    S = np.std(omega)

    V = S / M

    return V

def calcular_metricas_fase(x, fs):
    """
    Calcula V, M y S siguiendo las ecuaciones (7), (8) y (9) del paper.
    [cite: 197, 199, 200, 206]
    """
    dt = 1.0 / fs
    fase_unwrapped = np.unwrap(np.angle(hilbert(x)))

    omega = np.diff(fase_unwrapped) / dt

    M = np.mean(omega)
    S = np.std(omega)
    V = S / M

    return V, M, S

def test_hipotesis_espinozo(x, fs, n_surrogates=19):
    """
    Realiza el test de hipótesis univariado siguiendo a Espinoso (2022).
    Devuelve los valores individuales de los surrogates para graficar.
    """
    # 1. Calcular métricas de la señal original
    v_orig, m_orig, s_orig = calcular_metricas_fase(x, fs)

    # 2. Generar N surrogates (H0: Proceso lineal estacionario)
    surrogates = IAAFT(x, N=n_surrogates)

    v_surr, m_surr, s_surr = [], [], []

    # Calculamos métricas para cada surrogate
    for i in range(n_surrogates):
        v, m, s = calcular_metricas_fase(surrogates[:, i], fs)
        v_surr.append(v)
        m_surr.append(m)
        s_surr.append(s)

    # 3. Criterio de rechazo de dos colas:
    # Se rechaza si el original es MENOR que el mínimo O MAYOR que el máximo

    # Lógica para M (Velocidad de fase media)
    rechazo_M_bajo = m_orig < np.min(m_surr) # Caso típico en focales según Espinoso
    rechazo_M_alto = m_orig > np.max(m_surr) # Caso de velocidad inusualmente alta
    rechazo_H0_M = rechazo_M_bajo or rechazo_M_alto

    # Lógica para V (Coeficiente de variación)
    rechazo_V_bajo = v_orig < np.min(v_surr)
    rechazo_V_alto = v_orig > np.max(v_surr)
    rechazo_H0_V = rechazo_V_bajo or rechazo_V_alto

    return {
        "original": {"V": v_orig, "M": m_orig, "S": s_orig},
        "surrogates_stats": {
            "V_mean": np.mean(v_surr),
            "M_mean": np.mean(m_surr),
            "V_min": np.min(v_surr),
            "V_max": np.max(v_surr),
            "M_min": np.min(m_surr),
            "M_max": np.max(m_surr)
        },
        "lista_surr_V": v_surr,
        "lista_surr_M": m_surr,
        "rechazo_H0_M": rechazo_H0_M,
        "rechazo_H0_V": rechazo_H0_V,
        "detalle_rechazo": {
            "M_tipo": "bajo" if rechazo_M_bajo else ("alto" if rechazo_M_alto else "ninguno"),
            "V_tipo": "bajo" if rechazo_V_bajo else ("alto" if rechazo_V_alto else "ninguno")
        }
    }


def graficar_test_surrogates(res_test, metrica='V', nombre_senal="Señal AR"):
    """
    Crea un histograma de los valores de los surrogates y marca el valor original.

    Args:
        res_test (dict): El diccionario devuelto por la función test_hipotesis_espinozo.
        metrica (str): 'V' o 'M' para elegir qué métrica visualizar.
        nombre_senal (str): Etiqueta para la gráfica.
    """
    # Extraer datos
    val_orig = res_test['original'][metrica]
    # Necesitaremos que la función de test devuelva la lista completa de valores
    # de los surrogates para graficar la distribución.
    # (Asegúrate de guardar 'v_surr' o 'm_surr' en el dict de salida del test).
    val_surr = res_test['lista_surr_' + metrica]

    plt.figure(figsize=(8, 5))

    # Graficar la distribución de los surrogates (procesos lineales)
    sns.histplot(val_surr, kde=True, color='grey', label='Distribución Surrogates (H0)', alpha=0.6)

    # Marcar el valor de la señal original
    plt.axvline(val_orig, color='red', linestyle='--', linewidth=2, label=f'Original ({metrica}={val_orig:.4f})')

    # Títulos y etiquetas siguiendo el estilo del paper
    plt.title(f'Test de Hipótesis: {nombre_senal} ({metrica})')
    plt.xlabel(f'Valor de {metrica}')
    plt.ylabel('Frecuencia / Recuento')

    # Indicar si hubo rechazo
    rechazo = res_test[f'rechazo_H0_{metrica}']
    resultado_texto = "RECHAZADO (P < 0.05)" if rechazo else "NO RECHAZADO"
    plt.annotate(f'Resultado: {resultado_texto}',
                 xy=(0.05, 0.9), xycoords='axes fraction',
                 fontsize=12, fontweight='bold', color='red' if rechazo else 'black')

    plt.legend()
    plt.show()

def compare_signals(fases_unwrapped_1, fases_unwrapped_2, fs, nombre1="AR 1", nombre2="AR 2"):
    """
    Calcula métricas y compara estadísticamente dos grupos de señales.
    """
    dt = 1.0 / fs

    # Función interna para métricas
    def obtener_metricas(fase):
        omega = np.diff(fase) / dt
        m = np.mean(omega)
        s = np.std(omega)
        v = s / m
        return v, m, s

    # Si tienes múltiples épocas o realizaciones, calculamos para cada una
    # (Asumiendo que fases_unwrapped es una lista de arrays)
    v1 = [obtener_metricas(f)[0] for f in fases_unwrapped_1]
    v2 = [obtener_metricas(f)[0] for f in fases_unwrapped_2]

    # Test estadístico (Mann-Whitney U)
    stat, p_value = stats.mannwhitneyu(v1, v2)

    print(f"--- Resultados de Comparación ($V$) ---")
    print(f"Media {nombre1}: {np.mean(v1):.4f}")
    print(f"Media {nombre2}: {np.mean(v2):.4f}")
    print(f"Valor p: {p_value:.5f}")

    if p_value < 0.05:
        print("¡Resultado Significativo! Las señales presentan niveles de irregularidad distintos.")
    else:
        print("No se encontraron diferencias significativas.")

    return v1, v2



In [ ]:
phi_pairs = [0,0.6,0.9,0.99]
p_list = [1]

cutoff = 9.0
order = 4
fs = 100

jump_ratios = []
phase_stds = []

b, a = sg.butter(order, cutoff, fs=fs, btype='low')

for phi_start in phi_pairs:
    for p in p_list:
        x_stat = generate_ar(phi_start, n=2000)

        analytic = hilbert(x_stat)
        radius = np.abs(analytic)

        plt.figure(figsize=(12,3))
        plt.plot(radius)
        plt.axhline(np.percentile(radius, 5), color='r', ls='--', label='5% envelope')
        plt.title(f"Envelope — φ = {phi_start}")
        plt.ylabel("|z(t)|")
        plt.xlabel("Time")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

        x_real = np.real(analytic)
        x_imag = np.imag(analytic)

        N = 20
        x_real = x_real[5:N]
        x_imag = x_imag[5:N]

        plt.figure(figsize=(4,4))
        plt.plot(x_real, x_imag, lw=0.6)
        plt.scatter(x_real[0], x_imag[0], color='green', label='Start', s=40)
        plt.scatter(x_real[-1], x_imag[-1], color='red', label='End', s=40)
        plt.xlabel("Real part")
        plt.ylabel("Hilbert transform")
        plt.title(f"Analytic signal trajectory — φ = {phi_start}")
        plt.axis('equal')
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

        #x_filt = sg.filtfilt(b, a, x_stat)

        freqs, Pxx = sg.periodogram(
            x_stat,
            fs=fs,
            window='hann',
            scaling='density'
        )

        plt.figure(figsize=(12,4))
        plt.plot(freqs, 10*np.log10(Pxx + 1e-12))
        plt.xlabel("Frequency [Hz]")
        plt.ylabel("PSD [dB/Hz]")
        plt.title(f"Low-pass filtered PSD — nonstationary AR(1), φ(t)={phi_start}")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(12,4))
        plt.plot(x_stat, label='Filtered signal')
        plt.title(f'Filtered nonstationary AR(1), φ(t)={phi_start}')
        plt.legend()
        plt.tight_layout()
        plt.show()

        phase = np.angle(hilbert(x_stat))

        plt.figure(figsize=(12,4))
        plt.hist(phase, bins=100, density=True)
        plt.xlabel("Phase (rad)")
        plt.ylabel("Probability density")
        plt.title(f"Phase Distribution of A = {phi_start}")
        plt.grid(True)
        plt.show()

        phase_unwrapped = np.unwrap(phase)
        dphi = np.diff(phase_unwrapped)

        plt.figure(figsize=(12,4))
        plt.hist(dphi, bins=100)
        plt.xlabel("Δφ (rad)")
        plt.ylabel("Count")
        plt.title(f"Phase increment distribution — A={phi_start}")
        plt.grid(True)
        plt.show()

        plt.figure(figsize=(12,3))
        plt.plot(dphi, lw=0.6)
        plt.axhline(np.pi, color='r', ls='--')
        plt.axhline(-np.pi, color='r', ls='--')
        plt.title(f"Phase increments — φ = {phi_start}")
        plt.ylabel("Δφ")
        plt.xlabel("Time")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        jump_ratios.append(np.mean(np.abs(dphi) > np.pi/2))
        phase_stds.append(np.std(dphi))


        plt.figure(figsize=(12,4))
        plt.plot(dphi, linewidth=0.8)
        plt.xlabel("Time [samples]")
        plt.ylabel("Phase increment Δφ (rad)")
        plt.title(f"Phase increments vs time — φ={phi_start}")
        plt.grid(True)
        plt.show()


        #phase_stat = np.unwrap(np.angle(hilbert(x_stat)))

        #plt.figure(figsize=(12,4))
        #plt.plot(phase_stat, label='Phase', color='gray')
        #plt.title('Phase')
        #plt.xlabel('Time [samples]')
        #plt.ylabel('Phase [rad]')
        #plt.legend()
        #plt.tight_layout()
        #plt.show()

        omega = comute_omega(dphi, fs)

        plt.figure(figsize=(12,4))
        plt.plot(omega, label='Omega', color='red')
        plt.xlabel('Time [samples]')
        plt.ylabel('Angular frequency [rad/s]')
        plt.legend()
        plt.tight_layout()
        plt.show()

plt.figure(figsize=(7,4))
plt.plot(phi_pairs, jump_ratios, 'o-', label='Jump ratio (>π/2)')
plt.axhline(0.1, color='r', linestyle='--', label='Reliable threshold')
plt.xlabel("AR coefficient φ")
plt.ylabel("Fraction of large phase jumps")
plt.title("Phase unwrap reliability vs AR coefficient")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Two cases: small phase jump vs large phase jump
r = 1.0

theta1 = np.deg2rad(20)
theta2_small = np.deg2rad(60)    # < pi/2 jump
theta2_large = np.deg2rad(160)   # > pi/2 jump

z1 = r * np.exp(1j * theta1)
z2_small = r * np.exp(1j * theta2_small)
z2_large = r * np.exp(1j * theta2_large)

plt.figure(figsize=(6,6))

# Small jump
plt.arrow(0, 0, z1.real, z1.imag, head_width=0.05, color='blue', label='z(t)')
plt.arrow(0, 0, z2_small.real, z2_small.imag, head_width=0.05, color='green', label='z(t+1), small jump')

# Large jump
plt.arrow(0, 0, z2_large.real, z2_large.imag, head_width=0.05, color='red', label='z(t+1), large jump')

plt.scatter([0],[0], color='black')
plt.axhline(0,color='gray'); plt.axvline(0,color='gray')
plt.gca().set_aspect('equal')
plt.xlim(-1.2,1.2); plt.ylim(-1.2,1.2)
plt.title("Phase jumps in the complex plane")
plt.legend()
plt.grid(True)
plt.show()


# Phi and irregularity

In [ ]:
b1 = np.zeros(50)
b1[0] = 0.001
for i in range(1, 50):
    b1[i] = b1[i-1] * 1.15

phi_range = 1 - b1
log_arange = np.log(b1)

def analise_phi(fs, phi_range):
    v_values = []
    m_values = []
    s_values = []
    for phi in phi_range:
        x = generate_ar(phi, 2000)
        v, m, s = calcular_metricas_fase(x, fs)
        s_values.append(s)
        v_values.append(v)
        m_values.append(m)
    return np.array(v_values), np.array(m_values), np.array(s_values)

v_phi, m_phi, s_values = analise_phi(fs, phi_range)

fig, ax1 = plt.subplots(figsize=(12, 8))


color = 'tab:red'
#ax1.set_ylabel('Irregularity (V)', fontsize=14, color='black')
#ax1.plot(log_arange, v_phi, 'o-', color=color)
#ax1.tick_params(axis='y', labelsize=12, labelcolor='black')

#color = 'tab:green'
ax1.set_xlabel('log(1 - a)', fontsize=14)
ax1.set_ylabel('Standard derivation (S)', fontsize=14, color='black')
ax1.plot(log_arange, s_values, 'o-', color='green')
ax1.tick_params(axis='y', labelsize=14,labelcolor='black')

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Mean phase velocity (M)', fontsize=14, color='black')
ax2.plot(log_arange, m_phi, 's-', color='blue')
ax2.tick_params(axis='y',labelsize=14, labelcolor='black')


fig.tight_layout()
plt.show()

ax.plot(log_arange, v_phi,
        marker='o',
        linestyle='-',
        linewidth=2,
        markersize=5,
        color='black',
        label='M and S')

# Labels
ax.set_xlabel(r'$\log(1 - a)$', fontsize=14)
ax.set_ylabel('M and S', fontsize=14)

# Grid
ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)

# Ticks
ax.tick_params(axis='both', labelsize=12)

# Remove top/right spines for cleaner look
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Legend
ax.legend(fontsize=12)

plt.tight_layout()

# -----------------------------
# Save figure (important!)
# -----------------------------
plt.savefig("irregularity_vs_log_a.png", dpi=300, bbox_inches='tight')
plt.savefig("irregularity_vs_log_a.pdf", bbox_inches='tight')  # best for thesis

plt.show()

#No stacionality

In [ ]:
from scipy.signal import hilbert, butter, filtfilt, periodogram

# ---- Parameters ----
phi1_list = [0.999]
p_list = [1]
n = 2000
fs = 100  # example sampling frequency

cutoff = 9.0   # Hz
order = 4
b, a = butter(order, cutoff/(fs/2), btype='low')  # normalized freq

# ---- Non-stationary AR(1) generator ----
def generate_nonstationary_ar(phi_start, phi_end, n, seed=42):
    rng = np.random.default_rng(seed)
    x = np.zeros(n)
    eps = rng.normal(0, 1, n)
    phi_t = np.linspace(phi_start, phi_end, n)  # slowly varying phi

    for t in range(1, n):
        x[t] = phi_t[t] * x[t-1] + eps[t]
    return x

# ---- Analysis loop ----
for phi1 in phi1_list:
    for p in p_list:
        # Non-stationary: phi decreases to half its initial value
        x_stat = generate_nonstationary_ar(phi1, phi1*0.5, n=n)

        mean_x = np.mean(x_stat)
        print(f"Mean of non-stationary AR(1) with phi={phi1}: {mean_x:.4f}")

        # Low-pass filter
        #x_filt = filtfilt(b, a, x_stat)

        # Periodogram
        freqs, Pxx = periodogram(
            x_stat,
            fs=fs,
            window='hann',
            scaling='density'
        )

        plt.figure(figsize=(12,4))
        plt.plot(freqs, 10*np.log10(Pxx + 1e-12))
        plt.xlabel("Frequency [Hz]")
        plt.ylabel("PSD [dB/Hz]")
        plt.title(f"Low-pass filtered periodogram — Non-stationary AR(1), φ = {phi1}")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

        # Time series plot
        plt.figure(figsize=(12,4))
        plt.plot(x_stat, label='Filtered Non-stationary AR(1)')
        plt.title(f'Filtered Non-stationary AR(1) — φ = {phi1}')
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Phase computation
        phase_stat = np.unwrap(np.angle(hilbert(x_stat)))

        plt.figure(figsize=(12,4))
        plt.plot(phase_stat, label='Phase', color='gray')
        plt.title(f'Phase — Non-stationary AR(1), φ = {phi1}')
        plt.xlabel('Time [s]')
        plt.ylabel('Phase [rad]')
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Instantaneous angular frequency
        omega = np.diff(phase_stat) * fs  # approximate omega
        plt.figure(figsize=(12,4))
        plt.plot(omega, label='Omega', color='red')
        plt.xlabel('Time [s]')
        plt.ylabel('Angular frequency [rad/s]')
        plt.title(f'Omega — Non-stationary AR(1), φ = {phi1}')
        plt.legend()
        plt.tight_layout()
        plt.show()


#Periodogram

In [ ]:
# -----------------------------
# AR generator
# -----------------------------
def generate_ar(phi, n, sigma=1.0, seed=None):
    rng = np.random.default_rng(seed)
    x = np.zeros(n)
    eps = rng.normal(0, sigma, n)
    for t in range(1, n):
        x[t] = phi * x[t-1] + eps[t]
    return x

# -----------------------------
# Phase irregularity metric
# -----------------------------
def phase_irregularity(x):
    phase = np.unwrap(np.angle(hilbert(x)))
    dphi = np.diff(phase)
    return np.std(dphi)  # higher = more irregular

# -----------------------------
# Experiment
# -----------------------------
fs = 1.0
n = 4000
phi_list = [0.1, 0.4, 0.7, 0.9, 0.99]

irreg_vals = []

for phi in phi_list:
    x = generate_ar(phi, n, sigma=1.0)

    # Periodogram
    freqs, Pxx = sg.periodogram(x, fs=fs, window='hann', scaling='density')

    I = phase_irregularity(x)
    irreg_vals.append(I)

    # Plot PSD
    plt.figure(figsize=(8,4))
    plt.plot(freqs, 10*np.log10(Pxx + 1e-12))
    plt.title(f"Periodogram — AR(1), φ = {phi}")
    plt.xlabel("Frequency")
    plt.ylabel("PSD [dB]")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# -----------------------------
# Summary plot
# -----------------------------
plt.figure(figsize=(7,4))
plt.plot(phi_list, irreg_vals, 'o-')
plt.xlabel("φ")
plt.ylabel("Phase irregularity (std Δφ)")
plt.title("Irregularity vs AR memory")
plt.grid(True)
plt.tight_layout()
plt.show()


# Filtering

In [ ]:
# -----------------------------
# 1. Function to generate AR(1) process
# -----------------------------
def generate_ar1(phi, n_samples, sigma=1.0, seed=None):
    rng = np.random.default_rng(seed)
    x = np.zeros(n_samples)
    epsilon = rng.normal(0, sigma, n_samples)
    for t in range(1, n_samples):
        x[t] = phi * x[t-1] + epsilon[t]
    return x

# -----------------------------
# 2. Function to calculate local irregularity (V) using sliding windows
# -----------------------------
def calculate_local_v(signal, window_size=50):
    # V = std(x_t - x_{t-1})
    diffs = np.diff(signal)
    v_values = []
    for i in range(len(diffs) - window_size):
        window = diffs[i:i + window_size]
        v_values.append(np.std(window))
    return np.array(v_values)

# -----------------------------
# Study 1: Rapid step change in phi
# -----------------------------
def study_step_change():
    n = 2000
    # Abrupt change in phi: from 0.2 (low memory) to 0.99 (high memory)
    phi_sequence = np.concatenate([np.full(n//2, 0.2), np.full(n//2, 0.99)])
    signal = np.zeros(n)
    for t in range(1, n):
        signal[t] = phi_sequence[t] * signal[t-1] + np.random.normal(0, 1)

    v_metric = calculate_local_v(signal)

    # Compute phase using Hilbert transform
    analytic_signal = hilbert(signal)
    phase = np.unwrap(np.angle(analytic_signal))

    plt.figure(figsize=(12, 8))

    # Plot AR signal
    plt.subplot(311)
    plt.plot(signal, lw=0.5)
    plt.title("AR(1) Signal with Abrupt Step Change in φ (0.2 → 0.99)")
    plt.axvline(n//2, color='black', linestyle='--', label="Step change")
    plt.legend()

    # Plot irregularity
    plt.subplot(312)
    plt.plot(v_metric, color='red')
    plt.axvline(n//2, color='black', linestyle='--', label="Step change")
    plt.ylabel("Irregularity (V)")
    plt.legend()

    # Plot phase
    plt.subplot(313)
    plt.plot(phase, color='purple', lw=0.8)
    plt.axvline(n//2, color='black', linestyle='--', label="Step change")
    plt.ylabel("Phase [rad]")
    plt.xlabel("Time [samples]")
    plt.title("Phase Behavior")
    plt.legend()

    plt.tight_layout()
    plt.show()

# -----------------------------
# Study 2: Change detection / anomaly detection
# -----------------------------

# -----------------------------
# Study 3: Band-specific irregularity
# -----------------------------
def study_band_irregularity():
    from scipy.signal import butter, filtfilt
    import numpy as np
    import matplotlib.pyplot as plt

    def lowpass_filter(data, cutoff, fs, order=4):
        nyq = fs / 2
        b, a = butter(order, cutoff / nyq, btype='low')
        return filtfilt(b, a, data)

    fs = 200
    N = 1000
    signal = generate_ar1(0, 4000)[N:]
    nyq = fs / 2

    percentages = np.arange(0.03, 0.9, 0.04)   # bandwidth fractions
    log_p = np.log(percentages)

    V_vals = []
    M_vals = []
    S_vals = []

    for p in percentages:
        cutoff = p * nyq
        x_lp = lowpass_filter(signal, cutoff, fs)

        # ---- Remove first and last 5% ----
        trim = int(0.05 * len(x_lp))
        x_lp_trim = x_lp[trim:-trim]

        v, m, s = calcular_metricas_fase(x_lp_trim, fs)
        V_vals.append(v)
        M_vals.append(m)
        S_vals.append(s)

        plt.figure(figsize=(10,3))
        plt.plot(x_lp_trim, label=f"{int(p*100)}% of lowest frequencies", color='orange')
        plt.xlabel("Sample")
        plt.ylabel("Amplitude")
        plt.title(f"Low-pass Filtered Signal ({int(p*100)}%)")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    # ---- V vs log bandwidth ----
    plt.figure(figsize=(10,4))
    plt.plot(percentages, V_vals, 'o-', color='red')
    plt.xlabel("log(bandwidth fraction)")
    plt.ylabel("Phase irregularity V")
    plt.title("Phase irregularity vs log bandwidth")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # ---- M and S on same axis, log bandwidth ----
    plt.figure(figsize=(10,4))
    plt.plot(percentages, M_vals, 'o-', label='M', color='tab:blue')
    plt.plot(percentages, S_vals, 's-', label='S', color='tab:green')
    plt.xlabel("log(bandwidth fraction)")
    plt.ylabel("Metric value")
    plt.title("Phase metrics vs log bandwidth")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


study_band_irregularity()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, hilbert

def calcular_metricas_fase_jump_filtered(x, fs, jump_thresh=np.pi/2):
    """
    Compute phase irregularity V, ignoring large jumps > jump_thresh in the derivative
    """
    dt = 1.0 / fs
    fase_unwrapped = np.unwrap(np.angle(hilbert(x)))
    omega = np.diff(fase_unwrapped) / dt

    # Ignore large jumps
    omega_filtered = omega[np.abs(omega) <= jump_thresh/dt]

    M = np.mean(omega_filtered)
    S = np.std(omega_filtered)
    V = S / M
    return V, M, S


fs = 200
N = 1000
n_samples = 2000
cutoff_freqs = np.linspace(1, 100, 20)   # Low-pass cutoff frequencies
phi_values = np.linspace(0.0, 0.99, 20)  # AR(1) phi values (memory / correlation)

Z = np.zeros((len(phi_values), len(cutoff_freqs)))

for i, phi in enumerate(phi_values):
    signal = generate_ar(phi, n_samples)
    for j, fc in enumerate(cutoff_freqs):
        x_lp = lowpass_filter(signal, fc, fs)
        trim = int(0.05 * len(x_lp))  # trim first and last 5%
        x_lp_trim = x_lp[trim:-trim]
        V, M, S = calcular_metricas_fase_jump_filtered(x_lp_trim, fs)
        Z[i, j] = V

# 3D surface plot
X, Y = np.meshgrid(cutoff_freqs, phi_values)
fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, Z, cmap='viridis')
ax.set_xlabel('Low-pass cutoff frequency (Hz)')
ax.set_ylabel('AR(1) φ (correlation)')
ax.set_zlabel('Phase irregularity V')
ax.set_title('Phase irregularity V vs AR correlation and low-pass cutoff')
fig.colorbar(surf, shrink=0.5, aspect=5)
plt.show()


# Comparation between phi = 0  and phi = 0.99

In [ ]:
# -------------------
# Parameters
# -------------------
N = 5000
A = 0.99
np.random.seed(0)

# -------------------
# Generate signals
# -------------------
# White noise
wn = np.random.randn(N)

# AR(1) process
ar = np.zeros(N)
eps = np.random.randn(N)
for i in range(1, N):
    ar[i] = A * ar[i-1] + eps[i]

# -------------------
# Hilbert phase
# -------------------
phase_wn = np.unwrap(np.angle(hilbert(wn)))
phase_ar = np.unwrap(np.angle(hilbert(ar)))

# Instantaneous phase velocity
vel_wn = np.abs(np.diff(phase_wn))
vel_ar = np.abs(np.diff(phase_ar))

# -------------------
# Plot
# -------------------
plt.figure(figsize=(8, 4))
plt.plot(vel_wn[:1000], label="White noise", alpha=0.7)
plt.plot(vel_ar[:1000], label="AR(1), A = 0.99", alpha=0.7)
plt.xlabel("Time")
plt.ylabel("Instantaneous phase velocity")
plt.title("Phase velocity: White noise vs correlated AR(1)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
def ensemble_stats(a, n_realizations=50, T=1000):
    all_x = np.zeros((n_realizations, T))

    for i in range(n_realizations):
        all_x[i] = generate_ar1(a, T)

    mean = np.mean(all_x, axis=0)
    std = np.std(all_x, axis=0)

    return mean, std


a = 0.9
mean, std = ensemble_stats(a)

plt.figure(figsize=(10,5))
plt.plot(mean, label="Ensemble mean")
plt.fill_between(range(len(mean)), mean-std, mean+std, alpha=0.3, label="±1 std")

plt.title(f"Ensemble statistics for a = {a}")
plt.xlabel("Time step")
plt.ylabel("x(t)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def compute_v(x):
    # placeholder (replace with your real V definition)
    return np.std(x) / (np.mean(np.abs(x)) + 1e-8)


def v_vs_realizations(a, max_n=100, T=1000):
    v_vals = []

    for n in range(5, max_n, 5):
        vals = []
        for _ in range(n):
            x = generate_ar1(a, T)
            vals.append(compute_v(x))
        v_vals.append(np.mean(vals))

    return v_vals


a = 0.9
v_vals = v_vs_realizations(a)

plt.figure()
plt.plot(v_vals)
plt.title("Convergence of V with number of realizations")
plt.xlabel("Number of realizations (x5)")
plt.ylabel("V")
plt.show()

In [ ]:
from scipy.signal import butter, filtfilt

def lowpass(x, cutoff=0.1, fs=1.0, order=3):
    b, a = butter(order, cutoff, btype='low')
    return filtfilt(b, a, x)


a = 0.9
x = generate_ar1(a)

cutoffs = [0.05, 0.1, 0.2, 0.4]

plt.figure(figsize=(10,6))

for c in cutoffs:
    xf = lowpass(x, cutoff=c)
    plt.plot(xf, label=f"cutoff={c}")

plt.title(f"Low-pass filtering effect (a={a})")
plt.legend()
plt.show()

In [ ]:
a_values = np.linspace(0, 0.99, 20)
N = 50
T = 500

V_mean = []
V_std = []

def metric_v(x):
    return np.std(x)  # placeholder → pon tu V real

for a in a_values:
    vals = []
    for _ in range(N):
        x = generate_ar1(a, T)
        vals.append(metric_v(x))
    V_mean.append(np.mean(vals))
    V_std.append(np.std(vals))

plt.figure(figsize=(10,5))
plt.errorbar(a_values, V_mean, yerr=V_std, fmt='-o', capsize=3)

plt.title("Dependence of V on AR coefficient a")
plt.xlabel("a (temporal correlation)")
plt.ylabel("V (mean ± std over realizations)")
plt.show()

In [ ]:
from scipy.signal import butter, filtfilt

def lowpass(x, cutoff, order=3):
    b, a = butter(order, cutoff, btype='low')
    return filtfilt(b, a, x)

a = 0.9
x = generate_ar1(a)

cutoffs = [0.05, 0.1, 0.2, 0.4, 0.8]

plt.figure(figsize=(10,5))

for c in cutoffs:
    xf = lowpass(x, c)
    plt.plot(xf, label=f"cutoff={c}")

plt.title(f"Low-pass filtering effect (a = {a})")
plt.legend()
plt.show()

#visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# ---------- Model ----------
def generate_ar(phi, n, sigma=1, seed=None):
    rng = np.random.default_rng(seed)
    x = np.zeros(n)
    eps = rng.normal(0, sigma, n)
    x[0] = eps[0]

    for t in range(1, n):
        x[t] = phi * x[t - 1] + eps[t]

    return x


def phase_increments(x):
    analytic_signal = hilbert(x)
    phase = np.unwrap(np.angle(analytic_signal))
    return np.diff(phase)


# ---------- Data ----------
phis = [0.0, 0.3, 0.6, 0.99]
signals = [generate_ar(phi, 600, seed=42) for phi in phis]
letters = ["A", "B", "C", "D"]

# ---------- Style ----------
plt.style.use('seaborn-v0_8-white')

fig, axes = plt.subplots(2, 2, figsize=(9.5, 6), sharex=True, sharey=True)
axes = axes.ravel()

for i, (ax, x, phi, letter) in enumerate(zip(axes, signals, phis, letters)):

    dphi = phase_increments(x)

    ax.hist(
        dphi,
        bins=80,
        range=(-np.pi, np.pi),
        density=True,
        color="black",
        alpha=0.75,
        edgecolor="none"
    )

    ax.axvline(np.pi/2, color="black", ls="--", lw=1)
    ax.axvline(-np.pi/2, color="black", ls="--", lw=1)

    # ---------- panel label OUTSIDE data ----------
    ax.annotate(
        letter,
        xy=(0, 1), xycoords='axes fraction',
        xytext=(-14, 8), textcoords='offset points',
        fontsize=12, fontweight='bold',
        ha='left', va='bottom'
    )

    ax.set_xlim(-np.pi, np.pi)
    ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_xticklabels([r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])

    ax.grid(axis="y", linestyle=":", linewidth=0.6, alpha=0.6)
    ax.tick_params(labelsize=9)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # ---------- Y LABELS: 2 "Count" per row ----------
    if i % 2 == 0:   # left column only
        ax.set_ylabel("Probability density", fontsize=11)
    else:
        ax.set_ylabel("")

# shared X label
fig.supxlabel("Phase increment (rad/sample)", fontsize=11)

plt.tight_layout(pad=0.5)
plt.savefig("ar_phase_increments_consistent.png", dpi=300, bbox_inches="tight")
plt.savefig("ar_phase_increments_consistent.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# -----------------------------
# Parámetros (alta resolución)
# -----------------------------
fs = 200
n_samples = 2000

# Cutoff como porcentaje de Nyquist
cutoff_pcts = np.linspace(1, 99, 50)
phi_values = np.linspace(0.0, 0.99, 50)

Z = np.zeros((len(phi_values), len(cutoff_pcts)))

# -----------------------------
# Filtro
# -----------------------------
def lowpass_filter(data, cutoff, fs, order=4):
    nyq = fs / 2
    Wn = min(cutoff / nyq, 0.999)
    b, a = butter(order, Wn, btype='low')
    return filtfilt(b, a, data)

# ----------------------------------------------------------
# IMPORTANTE:
# calcular_metricas_fase() debe hacer:
#
# analytic = hilbert(x)
# trim = int(0.05 * len(analytic))
# analytic = analytic[trim:-trim]
#
# y calcular V, M, S usando esa señal analítica recortada.
# ----------------------------------------------------------

# -----------------------------
# Cómputo
# -----------------------------
for i, phi in enumerate(phi_values):

    signal = generate_ar(phi, n_samples)

    for j, pct in enumerate(cutoff_pcts):

        # % de Nyquist -> Hz
        fc = (pct / 100.0) * (fs / 2)

        # Filtrado
        x_lp = lowpass_filter(signal, fc, fs)

        # Las métricas se calculan sobre la señal completa.
        # El recorte del 5% se hace DESPUÉS de Hilbert
        # dentro de calcular_metricas_fase().
        V, M, S = calcular_metricas_fase(x_lp, fs)

        Z[i, j] = V

# -----------------------------
# Plot
# -----------------------------
plt.style.use('seaborn-v0_8-white')

fig, ax = plt.subplots(figsize=(10, 6.5), dpi=300)

im = ax.imshow(
    Z,
    aspect='auto',
    origin='lower',
    extent=[
        cutoff_pcts.min(),
        cutoff_pcts.max(),
        phi_values.min(),
        phi_values.max()
    ],
    cmap='viridis'
)

ax.set_xlabel('Low-pass cutoff (%)', fontsize=14)
ax.set_ylabel(r'AR coefficient $a$', fontsize=14)

cbar = plt.colorbar(im)
cbar.set_label('V', fontsize=14)
cbar.ax.tick_params(labelsize=12)

ax.tick_params(labelsize=12)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

plt.savefig("heatmap_tesis_V.png", dpi=300, bbox_inches='tight')
plt.savefig("heatmap_tesis_V.pdf", bbox_inches='tight')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Generate parameter range
# -----------------------------
b1 = np.zeros(50)
b1[0] = 0.001
for i in range(1, 50):
    b1[i] = b1[i-1] * 1.15

phi_range = 1 - b1
log_arange = np.log(b1)

# -----------------------------
# Analysis
# -----------------------------
def analise_phi(fs, phi_range):
    v_values, m_values, s_values = [], [], []

    for phi in phi_range:
        x = generate_ar(phi, 2000)
        v, m, s = calcular_metricas_fase(x, fs)
        v_values.append(v)
        m_values.append(m)
        s_values.append(s)

    return np.array(v_values), np.array(m_values), np.array(s_values)

v_phi, m_phi, s_values = analise_phi(fs, phi_range)

import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-white')

# ==========================================
# Compact paper figure
# ==========================================
fig, axes = plt.subplots(
    1, 2,
    figsize=(7.2, 3.0),
    dpi=300,
    sharex=True
)

# ------------------------------------------
# Panel A: Irregularity
# ------------------------------------------
ax = axes[0]

ax.plot(
    log_arange,
    v_phi,
    color='tab:red',
    marker='o',
    markersize=2.5,
    linewidth=0.9,
)

ax.set_xlabel(r'$\log(1-a)$', fontsize=9)
ax.set_ylabel('V', fontsize=9)

ax.grid(True, linestyle=':', linewidth=0.4, alpha=0.5)

ax.annotate(
    'A',
    xy=(0, 1),
    xycoords='axes fraction',
    xytext=(-12, 6),
    textcoords='offset points',
    fontsize=11,
    fontweight='bold'
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ------------------------------------------
# Panel B: Mean and Std
# ------------------------------------------
ax = axes[1]

ax.plot(
    log_arange,
    m_phi,
    color='tab:blue',
    marker='o',
    markersize=2.5,
    linewidth=0.9,
    label='M'
)

ax.plot(
    log_arange,
    s_values,
    color='tab:green',
    marker='s',
    markersize=2.5,
    linewidth=0.9,
    label='S'
)

ax.set_xlabel(r'$\log(1-a)$', fontsize=9)
ax.set_ylabel('Metric value', fontsize=9)

ax.grid(True, linestyle=':', linewidth=0.4, alpha=0.5)

ax.legend(
    frameon=False,
    fontsize=8,
    loc='best'
)

ax.annotate(
    'B',
    xy=(0, 1),
    xycoords='axes fraction',
    xytext=(-12, 6),
    textcoords='offset points',
    fontsize=11,
    fontweight='bold'
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for ax in axes:
    ax.tick_params(labelsize=8)

plt.tight_layout(pad=0.5, w_pad=1.2)

plt.savefig("metrics_vs_log_a_paper.pdf", bbox_inches='tight')
plt.savefig("metrics_vs_log_a_paper.png", dpi=300, bbox_inches='tight')

plt.show()
